In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"
# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)



In [2]:
# load the actual model to make sure it works
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embed via HuggingFace transformers (ground truth)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
s   = "Explain MoE in transformers in 3 sentences."
ids = processor.tokenizer(s, return_tensors="pt")["input_ids"]

embed_tokens = model.get_input_embeddings()          # the nn.Embedding itself
theirs       = embed_tokens(ids)

print(f"weight : {tuple(embed_tokens.weight.shape)}  dtype={embed_tokens.weight.dtype}")
print(f"ids    : {tuple(ids.shape)}  {ids[0, :10].tolist()}")
print(f"vecs   : {tuple(theirs.shape)}")
print(f"theirs[0, 0, :6] = {theirs[0, 0, :6].tolist()}")

weight : (262144, 1536)  dtype=torch.bfloat16
ids    : (1, 10)  [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761]
vecs   : (1, 10, 1536)
theirs[0, 0, :6] = [-0.11279296875, 0.18359375, -0.08251953125, -0.7265625, 2.171875, -1.484375]


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Multimodal embedding — local test_data (Moon.jpg + Chats.mp3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "path": "test_data/Moon.jpg"},
        {"type": "audio", "path": "test_data/Chats.mp3"},
        {"type": "text",  "text": "Describe the image and the audio."},
    ],
}]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

fused = out.hidden_states[0]   # fused residual stream: text + image + audio tokens

print("inputs keys :", list(inputs.keys()))
for k, v in inputs.items():
    if torch.is_tensor(v):
        print(f"  {k:20s} {tuple(v.shape)}")
print(f"fused       : {tuple(fused.shape)}  dtype={fused.dtype}")
print(f"fused[0,  0, :6] = {fused[0,  0, :6].tolist()}   # first token")
print(f"fused[0, -1, :6] = {fused[0, -1, :6].tolist()}   # last token")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/gemma4/feature_extraction_gemma4.py:208: RuntimeWarning: divide by zero encountered in matmul
  mel_spec = np.matmul(magnitude_spec, self.mel_filters)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/g

inputs keys : ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'input_features', 'input_features_mask']
  input_ids            (1, 415)
  attention_mask       (1, 415)
  mm_token_type_ids    (1, 415)
  pixel_values         (1, 2520, 768)
  image_position_ids   (1, 2520, 2)
  input_features       (1, 537, 128)
  input_features_mask  (1, 537)
fused       : (1, 415, 1536)  dtype=torch.bfloat16
fused[0,  0, :6] = [-1.640625, -1.53125, 0.1884765625, -1.484375, -0.99609375, -0.0272216796875]   # first token
fused[0, -1, :6] = [1.6015625, -1.828125, -0.36328125, 0.83203125, 0.0030364990234375, 0.014404296875]   # last token


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Extract image/audio soft-token slices from the fused residual stream
#  so we can compare against our projector outputs in load_pytorch_model.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tt = inputs["mm_token_type_ids"][0]      # (seq,)
uniq, counts = torch.unique(tt, return_counts=True)
print("mm_token_type_ids unique:", dict(zip(uniq.tolist(), counts.tolist())))

for t in uniq.tolist():
    if t == 0:          # text
        continue
    idx   = (tt == t).nonzero(as_tuple=True)[0]
    slice_ = fused[0, idx]     # (n_tokens, 1536)
    label  = {1: "image", 2: "audio", 3: "audio"}.get(t, f"type={t}")
    print(f"\n{label}  type_id={t}  → {tuple(slice_.shape)}  dtype={slice_.dtype}")
    print(f"  first token : {slice_[0, :6].tolist()}")


mm_token_type_ids unique: {0: 20, 1: 260, 3: 135}

image  type_id=1  → (260, 1536)  dtype=torch.bfloat16
  first token : [0.9453125, 0.1572265625, -0.88671875, -1.5546875, -0.859375, 0.314453125]

audio  type_id=3  → (135, 1536)  dtype=torch.bfloat16
  first token : [-1.90625, 0.037841796875, -3.8125, -1.3203125, -1.7578125, 0.8046875]


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  RMSNorm — apply HF's final norm to a fixed random tensor.
#  This is the ground truth our GemmaRMSNorm must match bit-for-bit.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

hf_norm = model.model.language_model.norm     # Gemma4RMSNorm(1536)

torch.manual_seed(0)
x = torch.randn(1, 8, 1536, dtype=torch.bfloat16)
y = hf_norm(x)

print(f"weight  : {tuple(hf_norm.weight.shape)}  dtype={hf_norm.weight.dtype}")
print(f"x       : {tuple(x.shape)}  dtype={x.dtype}")
print(f"y       : {tuple(y.shape)}  dtype={y.dtype}")
print(f"y[0,0,:6] = {y[0, 0, :6].tolist()}")

weight  : (1536,)  dtype=torch.bfloat16
x       : (1, 8, 1536)  dtype=torch.bfloat16
y       : (1, 8, 1536)  dtype=torch.bfloat16
y[0,0,:6] = [20.125, 2.59375, 14.125, 26.125, 7.59375, -20.5]


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  RoPE — pull HF's rotary module and emit (cos, sin) for both layer types.
#  This is the ground truth our GemmaRoPE must match bit-for-bit.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
hf_rope = model.model.language_model.rotary_emb

seq_len = 8
position_ids = torch.arange(seq_len, dtype=torch.long)[None]   # (1, S)

# global path: head_dim=512
x512 = torch.zeros(1, seq_len, 512, dtype=torch.bfloat16)
cos_g, sin_g = hf_rope(x512, position_ids, layer_type="full_attention")

# local path: head_dim=256
x256 = torch.zeros(1, seq_len, 256, dtype=torch.bfloat16)
cos_l, sin_l = hf_rope(x256, position_ids, layer_type="sliding_attention")

print(f"global cos: {tuple(cos_g.shape)}  cos[0,1,:6] = {cos_g[0, 1, :6].tolist()}")
print(f"global sin: {tuple(sin_g.shape)}  sin[0,1,:6] = {sin_g[0, 1, :6].tolist()}")
print(f"  → tail (no-rope dims) cos[0,1,-4:] = {cos_g[0, 1, -4:].tolist()}  (should be ~1)")
print(f"  → tail (no-rope dims) sin[0,1,-4:] = {sin_g[0, 1, -4:].tolist()}  (should be ~0)")
print()
print(f"local  cos: {tuple(cos_l.shape)}  cos[0,1,:6] = {cos_l[0, 1, :6].tolist()}")
print(f"local  sin: {tuple(sin_l.shape)}  sin[0,1,:6] = {sin_l[0, 1, :6].tolist()}")

global cos: (1, 8, 512)  cos[0,1,:6] = [0.5390625, 0.58203125, 0.625, 0.66015625, 0.69140625, 0.72265625]
global sin: (1, 8, 512)  sin[0,1,:6] = [0.83984375, 0.8125, 0.78125, 0.75, 0.72265625, 0.69140625]
  → tail (no-rope dims) cos[0,1,-4:] = [1.0, 1.0, 1.0, 1.0]  (should be ~1)
  → tail (no-rope dims) sin[0,1,-4:] = [0.0, 0.0, 0.0, 0.0]  (should be ~0)

local  cos: (1, 8, 256)  cos[0,1,:6] = [0.5390625, 0.59765625, 0.6484375, 0.69140625, 0.73046875, 0.765625]
local  sin: (1, 8, 256)  sin[0,1,:6] = [0.83984375, 0.80078125, 0.76171875, 0.72265625, 0.6796875, 0.640625]


In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Self-Attention — pull HF's layer-0 (local) and layer-4 (global) blocks
#  and run them on a fixed hidden tensor. This is the ground truth our
#  GemmaAttention must match bit-for-bit.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

torch.manual_seed(0)
B, S = 1, 8
hidden = torch.randn(B, S, 1536, dtype=torch.bfloat16)
position_ids = torch.arange(S, dtype=torch.long)[None]

# Additive causal mask of {0, -inf}, shape (1, 1, S, S). For S=8 this
# coincides with the sliding-window mask (window=512 > S), so both
# layer types take the same mask.
i = torch.arange(S)
allowed = i[:, None] >= i[None, :]
mask = torch.zeros(S, S, dtype=torch.float32).masked_fill(~allowed, float("-inf"))[None, None]

# ── local layer 0 (sliding_attention, head_dim=256) ─────────────
hf_l = model.model.language_model.layers[0].self_attn
x256 = torch.zeros(1, S, 256, dtype=torch.bfloat16)
cos_l, sin_l = hf_rope(x256, position_ids, layer_type="sliding_attention")

with torch.no_grad():
    out_l, _ = hf_l(
        hidden,
        position_embeddings=(cos_l, sin_l),
        attention_mask=mask,
        shared_kv_states={},
    )

print(f"local  out: {tuple(out_l.shape)}  dtype={out_l.dtype}")
print(f"  out[0, 0, :6] = {out_l[0, 0, :6].tolist()}")
print(f"  out[0,-1, :6] = {out_l[0,-1, :6].tolist()}")

# ── global layer 4 (full_attention, head_dim=512) ───────────────
hf_g = model.model.language_model.layers[4].self_attn
x512 = torch.zeros(1, S, 512, dtype=torch.bfloat16)
cos_g, sin_g = hf_rope(x512, position_ids, layer_type="full_attention")

with torch.no_grad():
    out_g, _ = hf_g(
        hidden,
        position_embeddings=(cos_g, sin_g),
        attention_mask=mask,
        shared_kv_states={},
    )

print(f"\nglobal out: {tuple(out_g.shape)}  dtype={out_g.dtype}")
print(f"  out[0, 0, :6] = {out_g[0, 0, :6].tolist()}")
print(f"  out[0,-1, :6] = {out_g[0,-1, :6].tolist()}")

local  out: (1, 8, 1536)  dtype=torch.bfloat16
  out[0, 0, :6] = [-0.169921875, 3.4375, -2.96875, -0.03759765625, 1.984375, -2.015625]
  out[0,-1, :6] = [-1.7421875, 1.140625, -0.9609375, 0.765625, 1.0, 1.984375]

global out: (1, 8, 1536)  dtype=torch.bfloat16
  out[0, 0, :6] = [2.71875, -2.96875, 1.09375, 3.859375, 0.123046875, -0.1708984375]
  out[0,-1, :6] = [1.3046875, 0.41015625, 0.373046875, 2.296875, -1.515625, 0.1962890625]
